# Urban100 with the one-step ADM posterior mean: baseline, Restormer on identical noise, saved pilots

Urban100 (100 images, about 1024 px) is the Gaussian colour set where supervised networks lead the diffusion prior.
DMID's released Urban100 outputs (one step at DMID's integer step) score 35.26 / 33.11 / 30.28 dB at σ 15 / 25 / 50;
GRL-B publishes 35.54 / 33.35 (σ 15 / 25) and Hi-IR 30.59 (σ 50). This run:

* measures ADM ×1 (one step at the continuous step matched to σ) and ADM ×2 (averaged with its horizontal flip) on all
  100 images, with Restormer's σ-specific models on the same noisy images;
* checks the image-size hypothesis on the 12 development images only: ADM ×1 on 512-px tiles (overlap 128) against
  the whole image;
* saves every ADM ×2 output (float16) as the pilot for CPU-side non-local Wiener filtering.

Split, fixed before any score: dev-12 = images 8, 16, …, 96 (tuning only); confirm-88 = the rest (claims).
Noise follows DMID's protocol (sample std exactly σ/255, unclipped) with the seeds of `diffusion-tweedie-benchmark`,
where Urban100 is the fourth set. Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

In [ ]:
import importlib.util, io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

SIGMAS = [15, 25, 50]
SEED_SETS = ["McMaster", "Kodak24", "CBSD68", "Urban100"]   # noise seeds as in diffusion-tweedie-benchmark
DEV12 = list(range(8, 100, 8))                                # images 8, 16, …, 96: tuning only
TILE, OVERLAP, TILE_BATCH = 512, 128, 2
TIME_LIMIT_H = 11.0
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, f"{OUT}/pilots"):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__)


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {path.split('/')[-1]}: {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


t0 = time.time()
GH = "https://github.com/{}/releases/download/{}/{}"
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": (GH.format("Li-Tong-621/DMID", "v1.0", "Datasets.zip"), 367_018_356),
}
for s in SIGMAS:
    FILES[f"restormer_{s}"] = (GH.format("swz30/Restormer", "v1.0", f"gaussian_color_denoising_sigma{s}.pth"), 104_611_957)
RESTORMER_SRC = f"{CACHE}/restormer_arch.py"
download("https://raw.githubusercontent.com/swz30/Restormer/68dc6ac472db26f16361150cb7a96a1bc87da93f/basicsr/models/archs/restormer_arch.py",
         RESTORMER_SRC)
with ThreadPoolExecutor(5) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TIMES["downloads"] = time.time() - t0
print(f"downloads {TIMES['downloads'] / 60:.1f} min")

## 1 · Urban100 from DMID's release, noise as in `diffusion-tweedie-benchmark`

In [ ]:
with zipfile.ZipFile(P["datasets"]) as z:
    names = sorted([n for n in z.namelist() if n.startswith("Datasets/Gaussion/Urban100/") and not n.endswith("/")], key=natkey)
    TEST = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB"))) for n in names]
assert len(TEST) == 100
print(len(TEST), "images; sizes from", min(im.shape[:2] for _, im in TEST), "to", max(im.shape[:2] for _, im in TEST))


def noise_seed(idx, sigma):
    return 1_000_000 * (SEED_SETS.index("Urban100") + 1) + 1000 * sigma + idx


def noisy_of(clean_u8, sigma, seed):
    """DMID's protocol: Gaussian noise normalised to sample std exactly sigma/255, added in float, never clipped."""
    x = torch.from_numpy(clean_u8).permute(2, 0, 1).float() / 255
    n = torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))
    return x + (sigma / 255) * n / n.std(unbiased=False)


_g = torch.exp(-torch.arange(-5, 6, dtype=torch.float64) ** 2 / (2 * 1.5 ** 2))
SSIM_WINDOW = (_g / _g.sum())[:, None] * (_g / _g.sum())[None, :]


def ssim(a, b):
    """skimage structural_similarity(gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=255)."""
    w = SSIM_WINDOW.to(a.device)[None, None].repeat(a.shape[0], 1, 1, 1)
    conv = lambda x: F.conv2d(x[None], w, groups=a.shape[0])[0]
    ux, uy = conv(a), conv(b)
    vx, vy, vxy = conv(a * a) - ux * ux, conv(b * b) - uy * uy, conv(a * b) - ux * uy
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    return float((((2 * ux * uy + c1) * (2 * vxy + c2)) / ((ux * ux + uy * uy + c1) * (vx + vy + c2))).mean())


def to_u8(x):
    """[1,3,H,W] float in [0,1] units -> rounded float [3,H,W] on the 0-255 scale."""
    return (x[0].clamp(0, 1) * 255).round()


def scores(out_u8, ref_u8, lp):
    a, b = out_u8.double(), ref_u8.double()
    mse = (a - b).pow(2).mean().item()
    with torch.no_grad():
        lpv = lp(a.float()[None] / 127.5 - 1, b.float()[None] / 127.5 - 1).item()
    return dict(psnr=10 * math.log10(255 ** 2 / mse) if mse > 0 else float("inf"), ssim=ssim(a, b), lpips=lpv)

## 2 · Networks on every GPU: ADM (fp16, SDPA on contiguous q, k, v), Restormer σ-specific (fp32), LPIPS

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def sdpa_attention(self, qkv):
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.transpose(1, 2).contiguous() for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(bs, -1, length)


gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)
RestormerNet = load_module("restormer_arch", RESTORMER_SRC).Restormer


def make_adm(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                     use_scale_shift_norm=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False).convert_to_fp16()
    return m


def make_restormers(device):
    nets = {}
    for s in SIGMAS:
        net = RestormerNet(inp_channels=3, out_channels=3, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4, heads=[1, 2, 4, 8],
                           ffn_expansion_factor=2.66, bias=False, LayerNorm_type="BiasFree", dual_pixel_task=False)
        net.load_state_dict(torch.load(P[f"restormer_{s}"], map_location="cpu", weights_only=True)["params"], strict=True)
        nets[s] = net.to(device).eval().requires_grad_(False)
    return nets


import lpips

CTX = [dict(dev=d, adm=make_adm(d), restormer=make_restormers(d), lpips=lpips.LPIPS(net="alex", verbose=False).to(d)) for d in DEVICES]
ADM_STATE.clear()

## 3 · Estimators

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()


def matched(sigma01):
    """Continuous step t and abar with abar = 1 / (1 + (2σ)^2)."""
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


def pad_to(y, k, mode="reflect"):
    h, w = y.shape[-2:]
    ph, pw = (-h) % k, (-w) % k
    return (F.pad(y, (0, pw, 0, ph), mode=mode) if ph or pw else y), (h, w)


@torch.no_grad()
def tweedie(model, y, t, ab):
    """One-step posterior mean of y [B,3,H,W] ([0,1] units, unclipped) at continuous step t with abar ab."""
    yp, (h, w) = pad_to(y, 64)
    x = math.sqrt(ab) * (2 * yp - 1)
    eps = model(x, torch.full((x.shape[0],), t, device=y.device))[:, :3].float()
    return (((x - math.sqrt(1 - ab) * eps) / math.sqrt(ab) + 1) / 2)[..., :h, :w]


@torch.no_grad()
def restormer_full(net, y):
    """Restormer's test script: reflect-pad to a multiple of 8, whole image."""
    yp, (h, w) = pad_to(y, 8)
    return net(yp)[..., :h, :w]


def tiled(fn, y, tile, overlap, batch=TILE_BATCH):
    """fn applied to overlapping tiles of y, blended with linear ramps."""
    _, _, H, W = y.shape
    stride = tile - overlap
    cover = lambda n: max(tile, tile + math.ceil(max(n - tile, 0) / stride) * stride)
    yp = F.pad(y, (0, cover(W) - W, 0, cover(H) - H), mode="reflect")
    r = torch.arange(tile, dtype=torch.float32, device=y.device)
    ramp = torch.clamp(torch.minimum(r + 1, tile - r) / overlap, max=1)
    w2 = ramp[:, None] * ramp[None, :]
    acc, wsum = torch.zeros_like(yp), torch.zeros_like(yp[:, :1])
    coords = [(i, j) for i in range(0, yp.shape[-2] - tile + 1, stride) for j in range(0, yp.shape[-1] - tile + 1, stride)]
    for b in range(0, len(coords), batch):
        cs = coords[b:b + batch]
        res = fn(torch.cat([yp[..., i:i + tile, j:j + tile] for i, j in cs]))
        for (i, j), out in zip(cs, res):
            acc[..., i:i + tile, j:j + tile] += out * w2
            wsum[..., i:i + tile, j:j + tile] += w2
    return (acc / wsum)[..., :H, :W]


def run_restormer(net, y):
    """Whole image; on out-of-memory, 512-px tiles with overlap 64 (recorded in the job info)."""
    try:
        return restormer_full(net, y), "whole"
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return tiled(lambda b: restormer_full(net, b), y, 512, 64, batch=1), "tiles"

## 4 · Every estimator on every image and σ (each one guarded); the ADM ×2 output is saved as a float16 pilot

In [ ]:
def run_image(ctx, job):
    idx, sigma = job
    name, clean = TEST[idx]
    dev, adm, lp = ctx["dev"], ctx["adm"], ctx["lpips"]
    split = "dev" if idx in DEV12 else "confirm"
    y = noisy_of(clean, sigma, noise_seed(idx, sigma))
    info = dict(image=name, idx=idx, sigma=sigma, split=split, height=clean.shape[0], width=clean.shape[1],
                noisy_sum=float(y.double().sum()))
    y = y[None].to(dev)
    ref = torch.from_numpy(clean).permute(2, 0, 1).to(dev).float()
    t, ab = matched(sigma / 255)
    outs, errors = {}, []

    def timed(label, fn):
        try:
            t0 = time.time()
            outs[label] = fn()
            torch.cuda.synchronize(dev)
            info[f"seconds {label}"] = time.time() - t0
        except Exception:
            errors.append((label, traceback.format_exc(limit=3)))
            torch.cuda.empty_cache()

    timed("ADM ×1", lambda: tweedie(adm, y, t, ab))
    timed("flipped", lambda: tweedie(adm, y.flip(-1), t, ab).flip(-1))
    if "ADM ×1" in outs and "flipped" in outs:
        outs["ADM ×2 (h-flip)"] = (outs["ADM ×1"] + outs["flipped"]) / 2
        np.savez_compressed(f"{OUT}/pilots/s{sigma}_{idx:03d}.npz", x=outs["ADM ×2 (h-flip)"][0].cpu().numpy().astype(np.float16))
    outs.pop("flipped", None)
    if split == "dev":
        timed("ADM ×1, 512-px tiles", lambda: tiled(lambda b: tweedie(adm, b, t, ab), y, TILE, OVERLAP))

    def restormer():
        out, mode = run_restormer(ctx["restormer"][sigma], y)
        info["restormer_mode"] = mode
        return out

    timed(f"Restormer σ{sigma}", restormer)
    info["peak_gib"] = torch.cuda.max_memory_allocated(dev) / 2 ** 30
    rows = [dict(image=name, idx=idx, sigma=sigma, split=split, method=m.replace(f" σ{sigma}", ""), **scores(to_u8(o), ref, lp))
            for m, o in outs.items()]
    return rows, info, errors


def worker_pool(jobs, fn):
    """Run fn(ctx, job) over jobs with one thread per GPU; returns (results, failures)."""
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))
                torch.cuda.empty_cache()

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    t0, last = time.time(), time.time()
    while any(th.is_alive() for th in threads):
        time.sleep(20)
        if time.time() - last > 600:
            last = time.time()
            print(f"  {len(results)}/{len(jobs)} done, {len(failures)} failed, {(time.time() - t0) / 60:.1f} min", flush=True)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
by_area = sorted(range(len(TEST)), key=lambda i: -TEST[i][1].shape[0] * TEST[i][1].shape[1])
smoke = [(by_area[0], 50), (DEV12[0], 25)]   # the largest image first: memory check before the full run
res, fail = worker_pool(smoke, run_image)
print("smoke test:", len(res), "ok,", len(fail), "failed;", [e[0] for _, _, errs in res for e in errs],
      [{k: (round(v, 2) if isinstance(v, float) else v) for k, v in info.items() if k.startswith(("seconds", "peak", "restormer"))}
       for _, info, _ in res])
jobs = [(i, s) for s in SIGMAS for i in range(len(TEST)) if (i, s) not in smoke]
res2, fail2 = worker_pool(jobs, run_image)
res, fail = res + res2, fail + fail2
SCORES = pd.DataFrame([r for rows, _, _ in res for r in rows])
INFO = pd.DataFrame([info for _, info, _ in res])
METHOD_ERRORS = [(info["idx"], info["sigma"], label, tb) for _, info, errs in res for label, tb in errs]
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
INFO.to_csv(f"{OUT}/image_info.csv", index=False)
TIMES["denoising"] = time.time() - t0
print(f"{len(res)}/{len(smoke) + len(jobs)} jobs, {len(fail)} failed jobs, {len(METHOD_ERRORS)} failed methods, {TIMES['denoising'] / 60:.1f} min")
for e in METHOD_ERRORS[:3]:
    print("METHOD ERROR", e)

## 5 · Results: means, and paired differences on identical noise

In [ ]:
mean = SCORES.groupby(["method", "sigma"])[["psnr", "ssim", "lpips"]].mean()
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()}, all 100 images")
    print(mean[metric].unstack().round(3 if metric == "psnr" else 4).to_string())
print("\nmean PSNR, confirm-88")
print(SCORES[SCORES.split == "confirm"].groupby(["method", "sigma"]).psnr.mean().unstack().round(3).to_string())
print("\nreference PSNR: DMID released 35.26 / 33.11 / 30.28; GRL-B 35.54 / 33.35 / 30.46; Hi-IR 35.46 / 33.34 / 30.59;"
      " Restormer published 35.13 / 32.96 / 30.02")
wide = SCORES.pivot_table(index=["idx", "sigma"], columns="method", values="psnr")
PAIRED = []
for a, b in [("ADM ×2 (h-flip)", "ADM ×1"), ("ADM ×1, 512-px tiles", "ADM ×1"), ("ADM ×1", "Restormer"), ("ADM ×2 (h-flip)", "Restormer")]:
    if a in wide and b in wide:
        for s, g in (wide[a] - wide[b]).dropna().groupby(level="sigma"):
            se = g.std(ddof=1) / math.sqrt(len(g))
            PAIRED.append(dict(contrast=f"{a} − {b}", sigma=s, mean=g.mean(), ci95=1.96 * se, better=int((g > 0).sum()), n=len(g)))
PAIRED = pd.DataFrame(PAIRED)
print("\npaired PSNR differences (dB)")
print(PAIRED.round(4).to_string(index=False))
PAIRED.to_csv(f"{OUT}/paired.csv", index=False)
print(INFO[[c for c in INFO if c.startswith(("seconds", "peak"))]].describe().round(2).to_string())
print("Restormer mode:", INFO.get("restormer_mode", pd.Series(dtype=str)).value_counts().to_dict())
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, means=mean.reset_index().to_dict("records"),
                   paired=PAIRED.to_dict("records"), dev12=DEV12, failed_jobs=[j for j, _ in fail],
                   failed_methods=[e[:3] for e in METHOD_ERRORS]), f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(len(os.listdir(f"{OUT}/pilots")), "pilots saved", f"total {(time.time() - T_START) / 60:.1f} min")